# Train Ticket — AI Anomaly Detection Demo

End-to-end walkthrough of the anomaly detector:

1. **Setup** — import modules and verify the tools are importable
2. **Baseline** — generate a normal-traffic snapshot and visualise the service dependency graph
3. **Anomaly** — simulate a 3 s latency injection on `ts-order-service`, score all services with Isolation Forest
4. **Results** — rank the top anomalous services in a table

> **No live cluster required.** All snapshots are generated by `simulate_traces.py`.
> To use real data, replace `simulate_snapshot()` calls with `collect_snapshot()` from `data_collector.py`.

## 1. Setup

Add the `src/` directory to Python's module search path, then import all libraries used in this notebook.

Required packages:
```
pip install matplotlib pandas networkx scikit-learn
```

In [ ]:
import os
import sys

# Resolve src/ relative to this notebook (works whether Jupyter is launched
# from notebooks/ or from the project root)
_nb_dir = os.path.abspath('.')
_src_dir = os.path.normpath(os.path.join(_nb_dir, '..', 'src'))
if not os.path.isdir(_src_dir):
    # Fallback: launched from project root
    _src_dir = os.path.normpath(os.path.join(_nb_dir, 'src'))
if _src_dir not in sys.path:
    sys.path.insert(0, _src_dir)

print(f'src/ path : {_src_dir}')
print(f'exists    : {os.path.isdir(_src_dir)}')

In [ ]:
import json
import warnings
warnings.filterwarnings('ignore')

import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.colors as mcolors
import pandas as pd
import networkx as nx

from simulate_traces import simulate_snapshot, generate_dataset, ANOMALY_SERVICE
from anomaly_detector import AnomalyDetector

print('matplotlib :', plt.matplotlib.__version__)
print('pandas     :', pd.__version__)
print('networkx   :', nx.__version__)
print()
print('simulate_traces  ✓')
print('anomaly_detector ✓')
print(f'Anomaly service  : {ANOMALY_SERVICE}')

## 2. Baseline — Normal Traffic

We generate **20 synthetic snapshots** under normal operating conditions using `generate_dataset()`,
then visualise the service dependency graph for the last snapshot.

Graph conventions:
- **Node size** — proportional to total inbound call count (busier = bigger)
- **Edge thickness** — proportional to average latency on that link
- **Node colour** — all green (no anomaly detected in baseline)

In [ ]:
# Generate 20 normal snapshots (seed fixed for reproducibility)
normal_snapshots = generate_dataset(n_normal=20, n_anomaly=0, seed=42)
baseline = normal_snapshots[-1]   # pick the last snapshot as the visual baseline

print(f'Normal snapshots generated : {len(normal_snapshots)}')
print(f'Baseline timestamp         : {baseline["collected_at"]}')
print(f'Services in baseline       : {len(baseline["nodes"])}')
print(f'Edges in baseline          : {len(baseline["edges"])}')
print()
print('Services:')
for node in sorted(baseline['nodes'], key=lambda n: n['service']):
    print(f'  {node["service"]:<35}  calls={node["call_count"]}')

In [ ]:
# ---------------------------------------------------------------------------
# Helper: build a NetworkX directed graph from a snapshot dict
# ---------------------------------------------------------------------------

def build_graph(snapshot):
    """Return a nx.DiGraph built from snapshot nodes and edges."""
    G = nx.DiGraph()
    for node in snapshot['nodes']:
        G.add_node(node['service'], call_count=node.get('call_count', 0))
    for edge in snapshot['edges']:
        G.add_edge(
            edge['source'], edge['target'],
            call_count=edge.get('call_count', 0),
            avg_latency_ms=edge.get('avg_latency_ms', 0.0),
        )
    return G


# ---------------------------------------------------------------------------
# Helper: draw the service dependency graph
# ---------------------------------------------------------------------------

def draw_service_graph(snapshot, title, node_scores=None, ax=None):
    """
    Draw the service dependency graph.

    Parameters
    ----------
    snapshot    : snapshot dict
    title       : plot title
    node_scores : dict { service_name: anomaly_score 0-1 }
                  If None, all nodes are drawn green.
    ax          : existing matplotlib Axes (optional; a new figure is created
                  when ax is None)
    """
    G = build_graph(snapshot)

    if ax is None:
        fig, ax = plt.subplots(figsize=(14, 8))
    else:
        fig = ax.figure

    # Deterministic spring layout (seed keeps it stable across calls)
    pos = nx.spring_layout(G, seed=7, k=2.5)

    # Node colours: green (score=0) through yellow to red (score=1)
    cmap = plt.get_cmap('RdYlGn_r')
    node_list = list(G.nodes())
    colors = [
        cmap(node_scores[n] if node_scores and n in node_scores else 0.0)
        for n in node_list
    ]

    # Node sizes proportional to inbound call count
    counts  = [G.nodes[n].get('call_count', 10) for n in node_list]
    max_cnt = max(counts) if counts else 1
    sizes   = [400 + 1600 * (c / max_cnt) for c in counts]

    # Edge widths proportional to average latency
    edges   = list(G.edges())
    lats    = [G[u][v].get('avg_latency_ms', 1.0) for u, v in edges]
    max_lat = max(lats) if lats else 1.0
    widths  = [0.8 + 4.0 * (l / max_lat) for l in lats]

    # Draw nodes
    nx.draw_networkx_nodes(
        G, pos, nodelist=node_list,
        node_color=colors, node_size=sizes,
        ax=ax, alpha=0.92,
    )
    # Draw labels
    nx.draw_networkx_labels(G, pos, font_size=7, ax=ax)
    # Draw edges
    nx.draw_networkx_edges(
        G, pos, edgelist=edges,
        width=widths, edge_color='steelblue',
        arrows=True, arrowsize=15, ax=ax,
        connectionstyle='arc3,rad=0.08',
    )
    # Edge latency labels
    edge_labels = {(u, v): f"{G[u][v]['avg_latency_ms']:.0f} ms" for u, v in edges}
    nx.draw_networkx_edge_labels(
        G, pos, edge_labels=edge_labels,
        font_size=6, ax=ax,
    )

    ax.set_title(title, fontsize=12, fontweight='bold', pad=12)
    ax.axis('off')

    # Colour bar (only when anomaly scores are provided)
    if node_scores is not None:
        sm = cm.ScalarMappable(
            cmap=cmap,
            norm=mcolors.Normalize(vmin=0, vmax=1),
        )
        sm.set_array([])
        fig.colorbar(sm, ax=ax, label='Anomaly Score', fraction=0.025, pad=0.02)

    return fig

In [ ]:
# Draw the baseline graph — all nodes are green (no anomaly)
draw_service_graph(
    baseline,
    title='Baseline — Normal Traffic (all nodes green)',
)
plt.tight_layout()
plt.show()

## 3. Anomaly — 3 s Delay on `ts-order-service`

We now simulate what happens after running `inject_anomaly.py`:
a **Chaos Mesh `NetworkChaos`** rule adds a **3 000 ms delay** to all
traffic leaving `ts-order-service`.

Steps:
1. Generate one anomalous snapshot with `simulate_snapshot(anomaly=True)`
2. Train an **Isolation Forest** on the 20 normal snapshots
3. Score every service in the anomalous snapshot
4. Redraw the graph — nodes are now coloured by their anomaly score

> The Isolation Forest is trained in **unsupervised** mode: it learns what
> "normal" looks like from the baseline data and flags any service whose
> feature vector deviates significantly.

In [ ]:
# --- Step 1: anomalous snapshot ---
anomalous_snapshot = simulate_snapshot(anomaly=True, seed=99)
print(f'Anomalous snapshot timestamp : {anomalous_snapshot["collected_at"]}')
print(f'Ground-truth anomaly service : {anomalous_snapshot["_meta"]["anomaly_service"]}')
print()

# Show how the injected delay is reflected in the snapshot
print('Edges involving ts-order-service:')
for edge in anomalous_snapshot['edges']:
    if 'order' in edge['target'] or 'order' in edge['source']:
        print(f'  {edge["source"]:<30} -> {edge["target"]:<30}  '
              f'avg_lat={edge["avg_latency_ms"]:.1f} ms  '
              f'calls={edge["call_count"]}')

In [ ]:
# --- Step 2: train the detector on normal data only ---
detector = AnomalyDetector(contamination=0.05)
detector.fit(normal_snapshots)

# --- Step 3: score the anomalous snapshot ---
results = detector.detect(anomalous_snapshot)

n_anomalies = sum(r['is_anomaly'] for r in results)
print(f'\nDetected {n_anomalies} anomalous service(s) out of {len(results)}.')
print()
print(f'{"Service":<35} {"Score":>8}  Anomaly?')
print('-' * 55)
for r in results:
    flag = '❌ ANOMALY' if r['is_anomaly'] else '✅ normal'
    print(f'{r["service"]:<35} {r["anomaly_score"]:>8.4f}  {flag}')

In [ ]:
# --- Step 4: draw the graph coloured by anomaly score ---
node_scores = {r['service']: r['anomaly_score'] for r in results}

draw_service_graph(
    anomalous_snapshot,
    title='Anomaly Active — 3 s Delay on ts-order-service\n'
          '(node colour: green = normal  →  red = anomalous)',
    node_scores=node_scores,
)
plt.tight_layout()
plt.show()

### Side-by-side comparison

The two graphs placed side by side make the impact of the anomaly immediately visible:
services that were green in the baseline turn orange or red when the 3 s delay is active.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(24, 9))

draw_service_graph(
    baseline,
    title='Baseline — Normal Traffic',
    node_scores=None,
    ax=axes[0],
)
draw_service_graph(
    anomalous_snapshot,
    title='Anomaly Active — 3 s Delay on ts-order-service',
    node_scores=node_scores,
    ax=axes[1],
)

fig.suptitle('Train Ticket — Service Dependency Graph', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

## 4. Results — Top Anomalous Services

The table below ranks every service by its anomaly score.
The **top 3** rows are highlighted in red — these are the services the on-call
engineer should investigate first.

Columns:
| Column | Description |
|---|---|
| Anomaly Score | 0 = normal, 1 = maximally anomalous |
| Avg Latency In (ms) | mean latency of all calls received by this service |
| Error Rate (%) | fraction of 5xx HTTP responses |
| Calls In / Out | inbound and outbound call counts in the snapshot window |

In [ ]:
df = pd.DataFrame([
    {
        'Service':              r['service'],
        'Anomaly Score':        r['anomaly_score'],
        'Is Anomaly':           'YES' if r['is_anomaly'] else 'no',
        'Avg Latency In (ms)':  round(r['features']['avg_latency_in_ms'], 1),
        'Error Rate (%)':       round(r['features']['error_rate'] * 100, 3),
        'Calls In':             int(r['features']['calls_in']),
        'Calls Out':            int(r['features']['calls_out']),
    }
    for r in results   # already sorted by anomaly_score descending
]).reset_index(drop=True)

# Highlight the top-3 rows (most anomalous)
def _highlight_top3(row):
    if row.name < 3:
        return ['background-color: #ffcccc; font-weight: bold'] * len(row)
    return [''] * len(row)

(
    df.style
    .apply(_highlight_top3, axis=1)
    .format({
        'Anomaly Score':       '{:.4f}',
        'Avg Latency In (ms)': '{:.1f}',
        'Error Rate (%)':      '{:.3f}',
    })
    .set_caption('Per-service anomaly scores — top-3 highlighted')
)

In [ ]:
# Text summary of the top-3 findings
print('=' * 60)
print('TOP-3 ANOMALOUS SERVICES')
print('=' * 60)
for rank, r in enumerate(results[:3], start=1):
    is_gt = (r['service'] == ANOMALY_SERVICE)
    gt_tag = '  <- injected fault' if is_gt else ''
    print(f'\n  #{rank}  {r["service"]}{gt_tag}')
    print(f'       anomaly_score      = {r["anomaly_score"]:.4f}')
    print(f'       avg_latency_in_ms  = {r["features"]["avg_latency_in_ms"]:.1f} ms')
    print(f'       error_rate         = {r["features"]["error_rate"]*100:.3f} %')
    print(f'       calls_in / out     = {int(r["features"]["calls_in"])} / '
          f'{int(r["features"]["calls_out"])}')
print()
print(f'Ground-truth anomaly service : {ANOMALY_SERVICE}')
caught = results[0]['service'] == ANOMALY_SERVICE
print(f'Detected as #1               : {"YES ✓" if caught else "NO ✗"}')